# Data Cleaning & Processing Module

Reads raw CSVs from `data/raw/` and produces cleaned, feature-engineered files in `data/cleaned/`.

Pipeline steps:
1. **Load** raw price, financial statement, and macro data
2. **Deduplicate** — detect and remove duplicate rows with logging
3. **Missing values** — forward-fill trading gaps; drop only unfillable head rows
4. **Data-type normalisation** — dates parsed, numerics coerced
5. **Outlier detection** — flag single-day moves > ±50% (possible splits / data errors)
6. **Feature engineering** — daily returns, 7-day & 30-day MAs, 30-day volatility, Bollinger Bands
7. **Save** cleaned files

In [ ]:
import pandas as pd
import numpy as np
import warnings
import logging
import os

warnings.filterwarnings("ignore")
logging.basicConfig(level=logging.INFO, format="%(message)s")
log = logging.getLogger()

RAW_DIR     = "data/raw"
CLEANED_DIR = "data/cleaned"
os.makedirs(CLEANED_DIR, exist_ok=True)

print("Data Cleaning & Processing Pipeline")
print("=" * 50)

## 1. Load Raw Price Data

print("Loading raw price data...")
price_df = pd.read_csv(f"{RAW_DIR}/sp500_prices.csv")

# Guard: 'Ticker' missing means the CSV was saved with the wrong stack level (yfinance 1.x bug)
if "Ticker" not in price_df.columns:
    raise ValueError(
        f"'Ticker' column not found in sp500_prices.csv.\n"
        f"Actual columns: {price_df.columns.tolist()}\n"
        "The CSV was generated by run_collect.py with the wrong yfinance 1.x stack level.\n"
        "Re-run 'run_collect.py' (now fixed) and retry this notebook."
    )

# Normalise date — strip timezone if present
price_df["Date"] = pd.to_datetime(price_df["Date"], utc=True).dt.tz_localize(None)
price_df = price_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

print(f"Loaded : {price_df.shape[0]:,} rows | {price_df['Ticker'].nunique()} tickers")
print(f"Range  : {price_df['Date'].min().date()} → {price_df['Date'].max().date()}")
price_df.head()

## 2. Price Data Cleaning

In [ ]:
print("=== Price Data Cleaning ===\n")
initial_rows = len(price_df)

# --- 2a. Duplicate removal ---
dupes = price_df.duplicated(subset=["Ticker", "Date"]).sum()
if dupes:
    price_df = price_df.drop_duplicates(subset=["Ticker", "Date"])
    log.info(f"[DEDUP]   Removed {dupes} duplicate rows")
else:
    log.info("[DEDUP]   No duplicates found")

# --- 2b. Enforce numeric types ---
for col in ["Open", "High", "Low", "Close", "Volume"]:
    if col in price_df.columns:
        price_df[col] = pd.to_numeric(price_df[col], errors="coerce")

# --- 2c. Forward-fill missing business days per ticker ---
# Explicit loop avoids pandas 3.x groupby.apply dropping the groupby-key column
all_bdates = pd.bdate_range(price_df["Date"].min(), price_df["Date"].max())

chunks = []
for ticker, group in price_df.groupby("Ticker"):
    group = group.set_index("Date").reindex(all_bdates).ffill()
    group.index.name = "Date"
    group["Ticker"] = ticker   # restore after reindex (new rows get NaN otherwise)
    chunks.append(group)

price_df = pd.concat(chunks).reset_index()
filled = len(price_df) - initial_rows
log.info(f"[FILL]    Forward-filled {filled:,} missing business-day entries")

# --- 2d. Drop rows where Close is still NaN (start of series) ---
null_close = price_df["Close"].isna().sum()
if null_close:
    price_df = price_df.dropna(subset=["Close"])
    log.info(f"[DROP]    Removed {null_close:,} rows with no Close price")

# --- 2e. Outlier flagging: single-day return > ±50% ---
price_df = price_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
pct_chg  = price_df.groupby("Ticker")["Close"].pct_change()
price_df["is_outlier"] = pct_chg.abs() > 0.50
n_outliers = price_df["is_outlier"].sum()
log.info(f"[OUTLIER] Flagged {n_outliers} entries with single-day move > ±50%")

if n_outliers:
    display(price_df[price_df["is_outlier"]][["Date", "Ticker", "Close"]].head(10))

print(f"\nPrice data after cleaning: {price_df.shape[0]:,} rows")

## 3. Feature Engineering

In [ ]:
print("=== Feature Engineering ===\n")
price_df = price_df.sort_values(["Ticker", "Date"])

g = price_df.groupby("Ticker", group_keys=False)

# Daily return
price_df["daily_return"] = g["Close"].pct_change().round(6)

# Rolling moving averages
price_df["ma7"]  = g["Close"].transform(lambda x: x.rolling(7,  min_periods=1).mean()).round(4)
price_df["ma30"] = g["Close"].transform(lambda x: x.rolling(30, min_periods=1).mean()).round(4)

# Annualised 30-day rolling volatility (std of daily returns × √252)
price_df["volatility_30d"] = (
    g["daily_return"]
    .transform(lambda x: x.rolling(30, min_periods=5).std() * np.sqrt(252))
    .round(6)
)

# Bollinger Bands — 20-day, ±2σ
bb_mid = g["Close"].transform(lambda x: x.rolling(20, min_periods=1).mean())
bb_std = g["Close"].transform(lambda x: x.rolling(20, min_periods=1).std())
price_df["bb_mid"]   = bb_mid.round(4)
price_df["bb_upper"] = (bb_mid + 2 * bb_std).round(4)
price_df["bb_lower"] = (bb_mid - 2 * bb_std).round(4)

# Cumulative return from each ticker's first available date
price_df["cum_return"] = (
    g["daily_return"]
    .transform(lambda x: (1 + x.fillna(0)).cumprod() - 1)
    .round(6)
)

print("Engineered features added:")
for feat, desc in [
    ("daily_return",   "% change in Close"),
    ("ma7",            "7-day rolling moving average"),
    ("ma30",           "30-day rolling moving average"),
    ("volatility_30d", "30-day annualised volatility"),
    ("bb_upper/mid/lower", "Bollinger Bands (20-day, ±2σ)"),
    ("cum_return",     "Cumulative return from first date"),
]:
    print(f"  {feat:<20} — {desc}")

print(f"\nFinal price shape: {price_df.shape}")
price_df[["Date","Ticker","Close","daily_return","ma7","ma30","volatility_30d"]].tail(8)

## 4. Financial Statements Cleaning

In [ ]:
print("=== Financial Statements Cleaning ===\n")

def clean_financials(path: str, name: str) -> pd.DataFrame:
    """Load, deduplicate, and normalise a financial statement CSV."""
    if not os.path.exists(path):
        print(f"  [SKIP] {name} not found at {path}")
        return pd.DataFrame()

    df = pd.read_csv(path)
    if df.empty:
        return df

    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
    df = df.dropna(subset=["Date"])

    dupes = df.duplicated(subset=["Ticker", "Date"]).sum()
    if dupes:
        df = df.drop_duplicates(subset=["Ticker", "Date"])
        log.info(f"  [{name}] Removed {dupes} duplicates")

    # Coerce all non-meta columns to numeric
    meta = {"Ticker", "Statement", "Date"}
    for col in [c for c in df.columns if c not in meta]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    df = df.sort_values(["Ticker", "Date"]).reset_index(drop=True)
    print(f"  [{name}] {df.shape[0]:,} rows | {df['Ticker'].nunique()} tickers")
    return df

income_df   = clean_financials(f"{RAW_DIR}/sp500_income.csv",   "Income")
balance_df  = clean_financials(f"{RAW_DIR}/sp500_balance.csv",  "Balance")
cashflow_df = clean_financials(f"{RAW_DIR}/sp500_cashflow.csv", "CashFlow")

## 5. Macro Data Cleaning

In [ ]:
print("=== Macro Data Cleaning ===\n")

macro_path = f"{RAW_DIR}/macro_market.csv"
if os.path.exists(macro_path):
    macro_df = pd.read_csv(macro_path)
    macro_df["Date"] = pd.to_datetime(macro_df["Date"], utc=True).dt.tz_localize(None)
    macro_df = macro_df.sort_values(["Ticker", "Date"]).reset_index(drop=True)

    # Coerce numeric columns
    meta_cols = {"Date", "Ticker", "Name"}
    for col in [c for c in macro_df.columns if c not in meta_cols]:
        macro_df[col] = pd.to_numeric(macro_df[col], errors="coerce")

    # Forward-fill weekends / holidays per instrument (explicit loop — pandas 3.x safe)
    macro_bdate_range = pd.bdate_range(macro_df["Date"].min(), macro_df["Date"].max())

    macro_chunks = []
    for ticker, group in macro_df.groupby("Ticker"):
        name_val = group["Name"].iloc[0] if "Name" in group.columns else None
        group = group.set_index("Date").reindex(macro_bdate_range).ffill()
        group.index.name = "Date"
        group["Ticker"] = ticker
        if name_val is not None:
            group["Name"] = name_val
        macro_chunks.append(group)

    macro_df = pd.concat(macro_chunks).reset_index()

    print(f"Macro data: {macro_df.shape[0]:,} rows | {macro_df['Ticker'].nunique()} instruments")
    macro_df.tail()
else:
    macro_df = pd.DataFrame()
    print("[SKIP] macro_market.csv not found — run DATACOLLECT first")

## 6. Save Cleaned Data

print("=== Saving Cleaned Data ===\n")

def _save(df: pd.DataFrame, filename: str):
    if df.empty:
        return
    path = f"{CLEANED_DIR}/{filename}"
    df.to_csv(path, index=False)
    print(f"  {filename:<30} — {df.shape[0]:,} rows saved")

_save(price_df,    "prices_clean.csv")
_save(income_df,   "income_clean.csv")
_save(balance_df,  "balance_clean.csv")
_save(cashflow_df, "cashflow_clean.csv")
_save(macro_df,    "macro_clean.csv")

print(f"\nAll cleaned files written to {CLEANED_DIR}/")